# Gold Medal Starmie: Prize Card Tracking

Hello! I reached the Gold Medal range in the Pokémon TCG AI Battle Challenge Simulation with a Starmie / Froslass rule-based agent.

First, huge thanks to ashleysandlin. My deck was based on their Starmie / Froslass list from Limitless:

https://play.limitlesstcg.com/tournament/69e4f71948d465883f718047/player/ashleysandlin/decklist

I should also be honest: I do not have a strong programming background, and most of the implementation was written with AI assistance. I also had not seriously played Pokémon TCG for about five years.

So my goal was simple:

**Use a simple deck, write simple rules, and make the agent execute them consistently.**

---

## Agent Overview

My agent is mostly rule-based.

I divided the logic into several modes:

* **Generic mode**: the default game plan and general scoring.
* **Matchup-specific modes**: special plans for common archetypes such as Lucario, Iono, Crustle, etc.
* **Finish mode**: a lethal-search mode that checks whether the agent can win this turn.

Most normal turns are handled by ordinary scoring rules: set up the board, attach Energy, evolve, attack, switch, or heal.

The matchup-specific modules change priorities depending on the opponent. For example, they can decide which attacker to focus on, which attack is preferred, what prize plan to follow, and when to heal or retreat.

Finally, when a winning line may exist, the agent uses Forward Search to ask:

> Can I win this turn?

If the search finds a verified winning sequence, the agent enters Finish mode and replays that line.

In short:

**Normal turns are handled by rules. Winning turns are verified by search.**

---

## Why Prize Tracking Matters

Search cards were one of the hardest problems.

Cards such as Hilda and Salvatore look inside the deck. If the agent gives Forward Search the wrong hidden deck information, the search may find an impossible plan.

For example, the simulation may think a card is still in the deck, find a winning line using it, and then the real game cannot reproduce that line because the card was actually prized.

I call this kind of failure a **NOMATCH**.

The fix was to imitate what a human player does after searching the deck: infer the Prize cards.

The tracker starts from the full deck list, subtracts every visible card, and if the remaining cards exactly match the number of Prize cards, it treats them as prized.

If anything is inconsistent, it returns unknown.

The key rule is:

**A wrong prize inference is worse than no prize inference.**

---

## The Important Detail: In-Flight Effect Cards

The trickiest bug was a card that is currently resolving an effect.

When Hilda is played, it leaves the hand. But while its search effect is still resolving, it may not yet be in the discard pile. For a short time, it is not in any ordinary public zone.

If we forget to subtract this "in-flight" card, the remaining count is off by one, and the Prize inference can become wrong.

The solution was:

`obs.select.effect`

During Hilda's search, `select.effect` points to Hilda.

By subtracting that card too, the tracker stays consistent across search-resolution frames.

This was more reliable than using only logs, because `obs.logs` only describes events since the previous selection. On later search frames, the original Play log may no longer be present.

---

## Prize Tracking Code

Below is the simplified reusable PrizeTracker.

It is not perfect, and it is probably not the cleanest way to do this. But it was useful for my rule-based agent, and I hope it helps others who are using search-based logic.

If you know a better implementation, please share it!

```python
from collections import Counter

class PrizeTracker:
    """Conservative Prize-card deduction reusable by any agent.

    The tracker deliberately returns unknown when the observation is ambiguous.
    A wrong prize check is more harmful than having no prize information at all.
    """

    def __init__(self, decklist):
        self._decklist = list(decklist)
        self._prized = None
        self._last_prize_count = None
        self._last_hand_by_serial = {}

    def update(self, obs, obs_dict=None):
        yi = obs.current.yourIndex
        player = obs.current.players[yi]
        prize_count = len(player.prize)
        hand_by_serial = {
            card.serial: card.id
            for card in player.hand or []
            if card is not None and getattr(card, "serial", None) is not None
        }

        # If Prize cards were taken, update the known Prize set.
        if (
            self._prized is not None
            and self._last_prize_count is not None
            and prize_count < self._last_prize_count
        ):
            taken = self._last_prize_count - prize_count
            card_ids = self._prize_to_hand(obs_dict, yi)
            # If logs do not explain the whole change, fall back to hand serial delta.
            if len(card_ids) != taken:
                card_ids = [
                    cid
                    for serial, cid in hand_by_serial.items()
                    if serial not in self._last_hand_by_serial
                ]
            # If the change is still ambiguous, forget the deduction.
            if len(card_ids) != taken or not self._remove(card_ids):
                self._prized = None

        self._last_prize_count = prize_count
        self._last_hand_by_serial = hand_by_serial

        # If we already know the Prize cards, keep that deduction.
        if self._prized is not None:
            return

        # We can only infer Prize cards when the deck is fully visible.
        if obs.select is None or obs.select.deck is None:
            return
        if len(obs.select.deck) != player.deckCount:
            return

        inferred = self._deduce(obs, player, yi)
        if inferred is not None:
            self._prized = inferred

    def _deduce(self, obs, player, player_index):
        remaining = Counter(self._decklist)

        def sub(card):
            if card is not None:
                remaining[card.id] -= 1

        # Visible deck during a search effect.
        for card in obs.select.deck:
            sub(card)
        # Hand.
        for card in player.hand or []:
            sub(card)
        # Active, bench, pre-evolutions, attached Energy, and tools.
        for pokemon in list(player.active or []) + list(player.bench or []):
            if pokemon is None:
                continue
            sub(pokemon)
            for c in getattr(pokemon, "preEvolution", None) or []:
                sub(c)
            for c in getattr(pokemon, "energyCards", None) or []:
                sub(c)
            for c in getattr(pokemon, "tools", None) or []:
                sub(c)
        # Discard pile.
        for card in player.discard or []:
            sub(card)
        # Stadium, if owned by this player.
        for card in obs.current.stadium or []:
            if card is not None and getattr(card, "playerIndex", None) == player_index:
                remaining[card.id] -= 1

        # The card currently resolving its effect.
        # Example: Hilda has left the hand, but may not yet be in discard.
        effect = getattr(obs.select, "effect", None)
        if effect is not None and getattr(effect, "playerIndex", None) == player_index:
            if remaining.get(effect.id, 0) > 0:
                remaining[effect.id] -= 1

        # Any inconsistency means this is not trustworthy.
        if any(count < 0 for count in remaining.values()):
            return None
        inferred = Counter({cid: count for cid, count in remaining.items() if count > 0})
        # The remaining cards must exactly match the number of Prize cards.
        if sum(inferred.values()) != len(player.prize):
            return None
        return inferred

    def _remove(self, card_ids):
        removals = Counter(card_ids)
        if any(self._prized.get(cid, 0) < count for cid, count in removals.items()):
            return False
        self._prized.subtract(removals)
        self._prized += Counter()  # Drop zero and negative entries.
        return True

    def _prize_to_hand(self, obs_dict, player_index):
        if not isinstance(obs_dict, dict):
            return []
        return [
            log["cardId"]
            for log in obs_dict.get("logs", [])
            if log.get("playerIndex") == player_index
            and log.get("fromArea") in (6, "PRIZE", "Prize")
            and log.get("toArea") in (2, "HAND", "Hand")
            and log.get("cardId") is not None
        ]

    def is_prized(self, card_id):
        """Return True / False when known, otherwise None."""
        if self._prized is None:
            return None
        return self._prized.get(card_id, 0) > 0

    def prized_cards(self):
        """Return Counter({card_id: count}) or None."""
        if self._prized is None:
            return None
        return self._prized.copy()
```

---

## Usage

```python
# At game start
tracker = PrizeTracker(decklist)

# Every frame
tracker.update(obs, obs_dict)

# Query
tracker.is_prized(card_id)      # True / False / None
tracker.prized_cards()          # Counter or None
```

---

## Closing Note

The main lesson for me was:

**Do not let Forward Search imagine a deck that the real game does not have.**

Good luck to everyone in the competition!
